# RealVisXL V4.0 High-Quality Professional Studio
Optimized steps, CFG scale, and negative prompts for stunning photorealistic output.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio

import torch
from diffusers import AutoPipelineForImage2Image, AutoPipelineForText2Image
import gradio as gr

print("Loading RealVisXL V4.0 Studio...")
pipe_t2i = AutoPipelineForText2Image.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe_t2i.enable_sequential_cpu_offload()

pipe_i2i = AutoPipelineForImage2Image.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe_i2i.enable_sequential_cpu_offload()

def generate(prompt, negative_prompt, steps, cfg):
    full_prompt = prompt + ", raw photo, highly detailed, 8k uhd, photorealistic, masterpiece, soft lighting"
    full_neg = negative_prompt + ", cartoon, illustration, painting, blurry, bad anatomy, deformed, low quality"
    image = pipe_t2i(prompt=full_prompt, negative_prompt=full_neg, num_inference_steps=int(steps), guidance_scale=float(cfg)).images[0]
    return image

def edit(image, prompt, negative_prompt, strength, steps, cfg):
    if image is None:
        return None
    full_prompt = prompt + ", raw photo, highly detailed, 8k uhd, photorealistic"
    full_neg = negative_prompt + ", blurry, low quality, deformed"
    img = pipe_i2i(prompt=full_prompt, negative_prompt=full_neg, image=image, strength=float(strength), num_inference_steps=int(steps), guidance_scale=float(cfg)).images[0]
    return img

demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Prompt", value="A cinematic portrait of a beautiful person, natural lighting, highly detailed"),
        gr.Textbox(label="Negative Prompt", value="blurry, bad anatomy"),
        gr.Slider(20, 50, value=30, step=1, label="Inference Steps"),
        gr.Slider(4.0, 10.0, value=7.5, step=0.5, label="CFG Scale")
    ],
    outputs=gr.Image(label="Generated Result"),
    title="RealVisXL V4.0 Pro Studio"
)
demo.launch(share=True, debug=True)